# Full Retrain — DimeNet+ Option B + MACE (notebook runner)

End-to-end reproduction of the `VAST_FULL_TRAIN_2MODELS` run, in one notebook.
Recommended GPU: >= 24 GB (RTX 4090/3090/A6000/A100 or 50-series), pytorch
CUDA image (>= CUDA 12.4). Run top-to-bottom. Each stage streams its live
output / progress bar into the cell.

Toggle which track to run near the top (`RUN_MACE`). Closing the browser is
fine mid-stage: kernels keep running on the instance.

## 0. Preflight
- `nvidia-smi` must show your GPU
- torch must see CUDA

In [ ]:
import shutil, subprocess, sys, os, textwrap
print(shutil.which('nvidia-smi'))

In [ ]:
import torch; print(torch.__version__, torch.version.cuda); print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no-gpu')

## Cell 2. Clone repo (first time) / update (re-run)
Runs in the notebook working dir (usually `/workspace`). Edit `PROJECT` if needed.


In [ ]:
PROJECT = '/workspace/training'
RUN_MACE = False          # True = run the MACE from-scratch track too
AQC = os.path.join(PROJECT, 'aqm-spice2')
AQM_SOL = os.path.join(PROJECT, 'AQM-sol-full.hdf5')
AQM_GAS = os.path.join(PROJECT, 'AQM-gas-full.hdf5')

In [ ]:
if not os.path.isdir(PROJECT):
    !git clone https://github.com/NghiemNgocDuc/training.git {PROJECT}
else:
    !git -C {PROJECT} pull
print('repo ready at', os.path.abspath(PROJECT))

In [ ]:
%cd {PROJECT}
# CPU Blackox / 50-series only: install cu128 wheels; older GPUs just `pip install torch`.
!pip install torch --index-url https://download.pytorch.org/whl/cu128
!pip install torch_geometric scikit-learn h5py numpy scipy pandas matplotlib tqdm
if RUN_MACE:
    !pip install mace-torch e3nn requests

## Cell 3. Download FULL AQM data (~3.0 GB from Zenodo) — skip if already present

In [ ]:
if not os.path.exists(AQM_SOL):
    !wget -O {AQM_SOL} 'https://zenodo.org/records/10208010/files/AQM-sol.hdf5?download=1'
if not os.path.exists(AQM_GAS):
    !wget -O {AQM_GAS} 'https://zenodo.org/records/10208010/files/AQM-gas.hdf5?download=1'
!ls -lh {AQM_SOL} {AQM_GAS}

## DimeNet+ track — Stage 1 (vacuum, from scratch on AQM-gas)
Full = 200 epochs / all data. Add `--k_folds 1 --max_structures 4000 --epochs 30` for a ~15 min sanity.

In [ ]:
STAGE1_OUT = os.path.join(AQC, 'pipeline', 'results_full')
CV_OUT = os.path.join(AQC, 'freesolv', 'cv_results_full')
!python aqm-spice2/pipeline/train_stage1_vacuum.py --hdf5 {AQM_GAS} --output_dir {STAGE1_OUT} --device cuda


## Stage 2 — correction (dG target on AQM-sol), init from stage1

In [ ]:
!python aqm-spice2/pipeline/train_stage2_correction.py --hdf5 {AQM_SOL} --gas_hdf5 {AQM_GAS} --vacuum_ckpt {STAGE1_OUT}/stage1_fold_1.pt --output_dir {STAGE1_OUT} --device cuda


## Stage 3 — FreeSolv 5-fold CV fine-tune (the actual retrain vs experiment)
Full: 200 epochs, patience 30, 5 conf-conformer test TTA.

In [ ]:
cv_ckpt = os.path.join(STAGE1_OUT, 'stage2_correction.pt')
!python aqm-spice2/freesolv/cv_finetune.py --conformers {os.path.join(PROJECT, 'freesolv_conformers.hdf5')} --cache_dir Data/FreeSolv --correction_ckpt {cv_ckpt} --output_dir {CV_OUT} --n_folds 5 --epochs 200 --patience 30 --n_conformers 5 --device cuda


## MACE track (optional - set RUN_MACE=True)

In [ ]:
if RUN_MACE:
    !python mace_freesolv/train_stage_a_scratch.py --hdf5_sol {AQM_SOL} --hdf5_gas {AQM_GAS} --device cuda
    !python mace_freesolv/main.py --device cuda --hdf5 {AQM_SOL}